NOTE: Running this code will not edit the existing files, it will make new files.

In [ ]:
# Standard library imports
import json
import os
import random
import re
import time
import requests
import pandas as pd
from urllib.parse import urlparse

# Selenium and WebDriver imports
from selenium import webdriver  # Main Selenium module to control the browser
from selenium.common.exceptions import StaleElementReferenceException
from selenium.webdriver.chrome.service import Service  # Manages the Chrome WebDriver service
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import Select, WebDriverWait

# Web scraping imports
from bs4 import BeautifulSoup
from fake_useragent import UserAgent

# WebDriver management
from webdriver_manager.chrome import ChromeDriverManager  # Auto-downloads & manages ChromeDriver
import chromedriver_autoinstaller


MAKE SURE CHROME WEBDRIVER IS WORKING PROPERLY:

common problem: webdriver old/corrupted/broken. delete with rm -rf ~/.wdm/drivers/chromedriver in terminal

In [ ]:
# Import necessary modules from Selenium and WebDriver Manager
from selenium import webdriver  # Main Selenium module to control the browser
from selenium.webdriver.chrome.service import Service  # Manages the Chrome WebDriver service
from webdriver_manager.chrome import ChromeDriverManager  # Auto-downloads & manages ChromeDriver

# Step 1: Install and set up the ChromeDriver
service = Service(ChromeDriverManager().install())  
# - ChromeDriverManager() automatically finds the right driver version for your Chrome browser
# - install() downloads it if it's not available
# - Service() starts a WebDriver service in the background

# Step 2: Configure Chrome options
options = webdriver.ChromeOptions()  # Create an options object for configuring Chrome
options.add_argument("--headless")  # Run Chrome in headless mode (no visible UI)
# - If you want to see the browser, remove the "--headless" argument

# Step 3: Initialize the Chrome WebDriver
driver = webdriver.Chrome(service=service, options=options)
# - This launches ChromeDriver with the configured options
# - The WebDriver service is started in the background

# Step 4: Open a webpage using Selenium
driver.get("https://www.google.com")  
# - This tells Selenium to navigate to Google.com

# Step 5: Confirm that the browser automation is working
print("✅ ChromeDriver is working correctly!")  
# - If this prints without error, Selenium is correctly installed & running

# Step 6: Close the WebDriver session
driver.quit()  
# - This closes the browser window and stops the WebDriver service
# - Always quit the driver when done to free up system resources


CREATE CLASSES, GRABBING CLASSES FROM EVERY DEPARTMENT FROM:
https://catalog.ucsd.edu/front/courses.html

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time
import json

# Set up Selenium
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run without opening a browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Base URL
BASE_URL = "https://catalog.ucsd.edu/front/courses.html"

# Open catalog homepage
print("🔍 Accessing UCSD course catalog...")
driver.get(BASE_URL)
time.sleep(2)  # Let the page load

# Get all department course links
department_links = driver.find_elements(By.XPATH, "//a[contains(text(), 'courses')]")
dept_urls = [link.get_attribute("href") for link in department_links]

print(f"✅ Found {len(dept_urls)} department course pages.")

all_courses = []

# Loop through each department page
for dept_url in dept_urls:
    print(f"📂 Scraping department page: {dept_url}")
    driver.get(dept_url)
    time.sleep(2)

    # Get page source and parse with BeautifulSoup
    soup = BeautifulSoup(driver.page_source, "html.parser")
    
    # Find all courses (adjust selectors if needed)
    course_elements = soup.find_all("p", class_="course-name")
    
    for course in course_elements:
        course_text = course.get_text(strip=True)
        print(f"🔹 Found course: {course_text}")  # Print raw text for debugging

        parts = course_text.split(".", 1)  # Split ID from name
        if len(parts) < 2:
            print("⚠️ Skipping malformed course entry.")
            continue
        
        course_id, course_name = parts
        description = course.find_next("p", class_="course-descriptions")
        description = description.get_text(strip=True) if description else "No description available"

        print(f"✅ Adding Course: ID={course_id.strip()}, Name={course_name.strip()}, Description={description[:100]}...")  # Truncate long descriptions

        all_courses.append({
            "course_id": course_id.strip().replace(" ",""),
            "course_name": course_name.strip(),
            "description": description
        })

# Close Selenium
driver.quit()

# Save to JSON
with open("courses_v1.json", "w") as f:
    json.dump(all_courses, f, indent=4)

print(f"🎉 Successfully scraped {len(all_courses)} courses and saved to 'coureses_v1.json'!")


TEST OPENING RATE MY PROFESSOR URL

In [ ]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from fake_useragent import UserAgent

# Install correct Chromedriver version
chromedriver_autoinstaller.install()

# Set up options
options = webdriver.ChromeOptions()
ua = UserAgent()
options.add_argument(f"user-agent={ua.random}")
options.add_argument("--disable-blink-features=AutomationControlled")

# Start WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Navigate to RateMyProfessors
BASE_URL = "https://www.ratemyprofessors.com/search/professors/1079?q=*"
driver.get(BASE_URL)

print("✅ Successfully opened RateMyProfessors!")

GET PROFESSOR DATA FROM RATE MY PROFESSOR

In [ ]:
import chromedriver_autoinstaller
import time
import json
import re
import random
import requests
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from fake_useragent import UserAgent
from bs4 import BeautifulSoup

# ✅ Install and configure WebDriver
chromedriver_autoinstaller.install()
options = webdriver.ChromeOptions()

# ✅ Random User-Agent to avoid bot detection
ua = UserAgent()
options.add_argument(f"user-agent={ua.random}")
options.add_argument("--disable-blink-features=AutomationControlled")

# ✅ Start WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# ✅ Navigate to UCSD RateMyProfessors page
BASE_URL = "https://www.ratemyprofessors.com/search/professors/1079?q=*"
print('opening url')
driver.get(BASE_URL)
WebDriverWait(driver, 10).until(
    EC.presence_of_element_located((By.TAG_NAME, "body"))
)
print("✅ Page loaded successfully!")
click_count = 0 

# ✅ Close pop-up if present
try:
    print("🔍 Checking for pop-up...")
    close_popup = WebDriverWait(driver, 10).until(
        EC.element_to_be_clickable((By.XPATH, "//button[contains(text(), 'Close')]"))
    )
    print(close_popup)
    close_popup.click()
    driver.execute_script("arguments[0].click();", close_popup) 
    print("✅ Pop-up closed successfully!")
    time.sleep(2)  
except:
    print("⚠️ No pop-up found or already closed.")

# Click "Show More" button until all professors are loaded
print("📜 Clicking 'Show More' buttons using JavaScript...")
while True:
    try:
        print('trying to find button')

        show_more_button = WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.XPATH, "//button[contains(text(), 'Show More')]"))
        )
        print(show_more_button)

        # ✅ Scroll to button to ensure it's in the viewport
        driver.execute_script("arguments[0].scrollIntoView(true);", show_more_button)
        driver.execute_script("arguments[0].click();", show_more_button) 

        #show_more_button.click()
        time.sleep(1)  # Small delay for stability
        
        click_count += 1
        print("✅ Clicked 'Show More' button.")

        # ✅ Wait to ensure professors load before clicking again
        time.sleep(random.uniform(3, 6))  # Random delay to mimic human behavior

        # ✅ Scroll to keep button in view
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")

    except:
        print("🚫 No more 'Show More' button found. Done clicking.")
        break  # Stop when no more buttons are found

# ✅ Get page source for scraping
soup = BeautifulSoup(driver.page_source, "html.parser")
professor_elements = soup.find_all("div", class_=re.compile(r"TeacherCard__InfoRatingWrapper.*"))

# ✅ Extract professor data
professors_data = []
for prof in professor_elements:
    try:
        # Extract name
        name_tag = prof.find("div", class_=re.compile(r"CardName__StyledCardName.*"))
        prof_name = name_tag.text.strip() if name_tag else "N/A"

        # Extract quality rating using regex
        quality_tag = prof.find("div", class_=re.compile(r"CardNumRating__CardNumRatingNumber.*"))
        quality = quality_tag.text.strip() if quality_tag else "N/A"

        # Extract number of ratings
        num_ratings_tag = prof.find("div", class_=re.compile(r"CardNumRating__CardNumRatingCount.*"))
        num_ratings = num_ratings_tag.text.strip() if num_ratings_tag else "N/A"

        # Extract "Would Take Again" percentage
        take_again_tag = prof.find("div", class_=re.compile(r"CardFeedback__CardFeedbackNumber.*"))
        would_take_again = take_again_tag.text.strip() if take_again_tag else "N/A"

        # Extract level of difficulty
        difficulty_tag = prof.find_all("div", class_=re.compile(r"CardFeedback__CardFeedbackNumber.*"))
        difficulty = difficulty_tag[1].text.strip() if len(difficulty_tag) > 1 else "N/A"

        # Extract department
        department_tag = prof.find("div", class_=re.compile(r"CardSchool__Department.*"))
        department = department_tag.text.strip() if department_tag else "N/A"

        # Extract university name
        university_tag = prof.find("div", class_=re.compile(r"CardSchool__School.*"))
        university = university_tag.text.strip() if university_tag else "N/A"

        # Extract profile link
        profile_link_tag = prof.find_parent("a")
        profile_link = "https://www.ratemyprofessors.com" + profile_link_tag["href"] if profile_link_tag else "N/A"

        # ✅ Open profile page to get courses
        profile_page = requests.get(profile_link, headers={"User-Agent": ua.random})
        prof_soup = BeautifulSoup(profile_page.text, "html.parser")
        course_elements = prof_soup.find_all("div", class_=re.compile(r"^RatingHeader__StyledClass"))
        prof_courses = set([course.text.strip() for course in course_elements if course.text.strip()])

        # ✅ Store professor data
        professor_info = {
            "name": prof_name,
            "quality_rating": quality,
            "num_ratings": num_ratings,
            "would_take_again": would_take_again,
            "difficulty": difficulty,
            "department": department,
            "university": university,
            "profile_link": profile_link,
            "professor_courses": list(prof_courses)  # Convert set to list
        }
        professors_data.append(professor_info)

        print(f"✅ Scraped {prof_name} | Quality: {quality} | {num_ratings} ratings | Take Again: {would_take_again}% | Difficulty: {difficulty}")

    except Exception as e:
        print(f"❌ Error extracting professor details: {e}")

# ✅ Save data to JSON file
with open("professors_v1.json", "w") as f:
    json.dump(professors_data, f, indent=4)

print(f"✅ Successfully saved {len(professors_data)} professors to 'professors_v1.json'!")

# ✅ Close WebDriver
driver.quit()


EDIT JSON

In [ ]:
import json
import re

def format_course_id(course_id):
    """Ensure consistent formatting of course IDs by adding space between letters and numbers."""
    return re.sub(r'([A-Za-z]+)(\d+)', r'\1 \2', course_id).upper()

# Load JSON files
courses_file_path = "./courses_v1.json"
professors_file_path = "./professors_v1.json"

with open(courses_file_path, "r") as f:
    courses_data = json.load(f)

with open(professors_file_path, "r") as f:
    professors_data = json.load(f)

# Standardize all course IDs in courses_data
course_dict = {format_course_id(course["course_id"]): course for course in courses_data}

print("Standardized Course IDs:", course_dict.keys())  # Debugging step

# Add professors to corresponding courses
for professor in professors_data:
    for course_id in professor["professor_courses"]:
        edited_course_id = format_course_id(course_id)  # Format professor course ID
        if edited_course_id in course_dict:
            if "professors" not in course_dict[edited_course_id]:
                course_dict[edited_course_id]["professors"] = []
            
            course_dict[edited_course_id]["professors"].append({
                "name": professor["name"],
                "quality_rating": professor["quality_rating"],
                "num_ratings": professor["num_ratings"],
                "would_take_again": professor["would_take_again"],
                "difficulty": professor["difficulty"],
                "profile_link": professor["profile_link"],
                "department": professor["department"]
            })
        else:
            print(f"Warning: Course ID {edited_course_id} not found in course_dict")  # Debugging step

# Convert back to a list
merged_courses = list(course_dict.values())

# Save the merged data
merged_file_path = "./courses_and_professors_v1.json"
with open(merged_file_path, "w") as f:
    json.dump(merged_courses, f, indent=4)

print("Merging complete. Saved to:", merged_file_path)


ADD PREREQUISITES TO JSON

In [ ]:
import json
import re

# ✅ Load existing course data
with open("courses_and_professors_v1.json", "r") as f:
    courses = json.load(f)

# ✅ Function to extract credits from course name
def extract_credits(course_name):
    match = re.search(r"\(([\d\sor]+)\)$", course_name)  # Matches (2 or 4) format
    return match.group(1) if match else "N/A"

# ✅ Function to extract prerequisites from description
def extract_prerequisites(description):
    match = re.search(r"Prerequisites:(.*)", description)  # Finds "Prerequisites:" and captures the text after it
    return match.group(1).strip() if match else "None"

# ✅ Process courses and add new attributes
for course in courses:
    course["credits"] = extract_credits(course["course_name"])
    course["prerequisites"] = extract_prerequisites(course["description"])

# ✅ Save updated data to a new JSON file
with open("courses_and_professors_v2.json", "w") as f:
    json.dump(courses, f, indent=4)

print("✅ Successfully added prerequisites and credits to the JSON file!")


TEST TO OPEN SCHEDULE OF CLASSES WEBSITE: [https://act.ucsd.edu/scheduleOfClasses/scheduleOfClassesStudent.htm] THIS DOC IS VERY USEFUL, HAS ALL THE INFORMATION ON COURSE OFFERINGS FOR EVERY DEPARTMENT FROM SUMMER 2023 - SPRING 2025

In [ ]:
import chromedriver_autoinstaller
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from fake_useragent import UserAgent

# Install correct Chromedriver version
chromedriver_autoinstaller.install()

# Set up options
options = webdriver.ChromeOptions()
ua = UserAgent()
options.add_argument(f"user-agent={ua.random}")
options.add_argument("--disable-blink-features=AutomationControlled")

# Start WebDriver
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Navigate to RateMyProfessors
BASE_URL = "https://act.ucsd.edu/scheduleOfClasses/scheduleOfClassesStudent.htm"
driver.get(BASE_URL)

print("✅ Successfully opened schedule!")

SCRAPE DATA FROM SCHEDULE OF CLASSES, FOR COURSE OFFERING TIMES

In [ ]:
import re
import json
import random
import time
from selenium.common.exceptions import StaleElementReferenceException
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup
from selenium.webdriver.support.ui import Select


# Set up Selenium WebDriver with User-Agent Spoofing
options = webdriver.ChromeOptions()
#options.add_argument("--headless")  # Run in headless mode (no UI)
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36")
service = Service(ChromeDriverManager().install())
driver = webdriver.Chrome(service=service, options=options)
wait = WebDriverWait(driver, 15)  # Increased wait time to handle slow loading

# Dictionary to store department-wise courses
all_courses_by_department = {}



for quarter in ['Spring Quarter 2024']:
    try:
        base_url = "https://act.ucsd.edu/scheduleOfClasses/scheduleOfClassesStudent.htm"

        # Step 1: Open the course schedule website
        driver.get(base_url)
        time.sleep(random.uniform(3, 5))  # Random sleep to avoid bot detection

        # Step 2: Select Fall 2023 from the term dropdown
        # term_dropdown = wait.until(EC.presence_of_element_located((By.NAME, "selectedTerm")))
        # select_term = Select(term_dropdown)
        #select_term.select_by_visible_text("Fall Quarter 2023")
        time.sleep(random.uniform(2, 4))  # Random delay

        # Step 3: Get all department options
        department_list = wait.until(EC.presence_of_element_located((By.NAME, "selectedSubjects")))
        departments = department_list.find_elements(By.TAG_NAME, "option")

        for dept_index in range(0,len(departments)):
            # **Re-Navigate to the main page before selecting the next department**
            driver.get(base_url)
            time.sleep(random.uniform(3, 5))
            term_dropdown = wait.until(EC.presence_of_element_located((By.NAME, "selectedTerm")))
            select_term = Select(term_dropdown)
            select_term.select_by_visible_text(quarter)


            # **Re-fetch the department list to avoid stale element reference**
            department_list = wait.until(EC.presence_of_element_located((By.NAME, "selectedSubjects")))
            departments = department_list.find_elements(By.TAG_NAME, "option")

            # Get department code and name
            #dept_code = departments[dept_index].text.split(" ")[0]  # Extract department code (e.g., "DSC")

            retry_attempts = 3  # Max retries for stale element
            for attempt in range(retry_attempts):
                try:
                    departments = department_list.find_elements(By.TAG_NAME, "option")  # Re-fetch to avoid staleness
                    dept_code = departments[dept_index].text.split(" ")[0]  # Extract department code
                    break  # Exit retry loop if successful
                except StaleElementReferenceException:
                    print(f"⚠️ Stale element detected. Retrying... ({attempt+1}/{retry_attempts})")
                    time.sleep(1)  # Wait briefly before retrying
            else:
                print(f"❌ Failed to retrieve department code for index {dept_index}")
                continue  # Skip this department if all attempts fail
            print(f"\n🔹 Scraping department: {dept_code}")

            # Select the department
            print(dept_code)
            #departments[dept_index].click()
            # dep_button = driver.find_element(By.ID, dept_code)
            # driver.execute_script("arguments[0].click();", dep_button)
            # time.sleep(random.uniform(1, 3))  # Random delay
            time.sleep(2)
            department_list = wait.until(EC.presence_of_element_located((By.NAME, "selectedSubjects")))
            time.sleep(2)
            for option in department_list.find_elements(By.TAG_NAME, "option"):
                if dept_code in option.text:
                    print(option.text)
                    print(f'dept_code: {dept_code} recogznied')
                    option.click()
                    driver.execute_script("arguments[0].click();", option)
                    break
            time.sleep(1)

            # Step 4: Click the Search button
            search_button = driver.find_element(By.ID, "socFacSubmit")
            driver.execute_script("arguments[0].click();", search_button)
            print("✅ Clicked search button")

            # Step 5: Wait for the results to load
            time.sleep(random.uniform(4, 7))

            # Step 6: Find the total number of pages for the department
            soup = BeautifulSoup(driver.page_source, "html.parser")

            page_numbers = []
            for a in soup.find_all("a", href=True):
                if "page=" in a["href"]:
                    page_num = a.text.strip()
                    if page_num.isdigit():
                        page_numbers.append(int(page_num))

            total_pages = max(page_numbers) if page_numbers else 1
            print(f"📄 Total pages found for {dept_code}: {total_pages}")

            dept_courses = [] # Use a set to avoid duplicate courses

            # Step 7: Loop through each page
            for page in range(1, total_pages + 1):
                # If we're not on the first page, navigate to the correct page
                if page > 1:
                    next_page_url = f"https://act.ucsd.edu/scheduleOfClasses/scheduleOfClassesStudentResult.htm?page={page}"
                    print(f"📌 Navigating to: {next_page_url}")
                    driver.get(next_page_url)
                    time.sleep(random.uniform(3, 6))  # Random delay

                # Step 8: Extract Course Numbers Properly
                soup = BeautifulSoup(driver.page_source, "html.parser")

                # for crsheader in soup.find_all("td", class_="crsheader"):
                #     course_number = crsheader.text.strip()
                    
                #     formatted_course = f"{dept_code} {course_number}"  # Append dept code before the number
                #     dept_courses.add(formatted_course.strip())  # Use set to remove duplicates
                last_course_number = None  # Track the last valid course number

                for row in soup.find_all("tr"):  # Iterate over table rows
                    crsheaders = row.find_all("td", class_="crsheader")  # Find all course number columns

                    for crsheader in crsheaders:
                        course_text = crsheader.text.strip()

                        # ✅ Capture 1, 2, or 3-digit course numbers with optional letter suffixes (e.g., "1", "10A", "197DC")
                        match = re.search(r'\b\d{1,3}[A-Z]*\b', course_text)  # Matches "1", "10", "101", "197DC", "20A"

                        if match:
                            course_number = match.group(0)  # Extract course number

                            # ✅ Ensure unique courses are stored properly
                            formatted_course = f"{dept_code} {course_number}"
                            dept_courses.append(formatted_course)


            # Store courses by department
            all_courses_by_department[dept_code] = list(dept_courses)  # Convert back to list for display
            
            # ✅ Print extracted courses for department
            print(f"\n📚 Extracted Courses for {dept_code}:")
            print(set(dept_courses))
            print("Length of dept_courses: " + str(len(dept_courses)))
            
            #file called _course_offerings.json must exist - this read file code is to deal with duplicates
            with open("course_offerings.json", "r") as f:
                course_data = json.load(f) 

            existing_courses = list(set(course_data[quarter]))  # Get existing courses for the department
            new_courses = list(set(dept_courses))  # New scraped courses

            # Combine old and new courses, removing duplicates
            course_data[quarter] = existing_courses + new_courses

            with open("course_offerings.json", "w") as f:
                json.dump(course_data, f, indent = 4)
    finally:
        driver.quit()  # Close the browser session

# ✅ Print all extracted courses in the end
print("\n🎯 Final Extracted Courses Across All Departments:")
for dept, courses in all_courses_by_department.items():
    print(f"\n🔹 {dept}: {len(courses)} courses")
    for course in courses:
        print(course)


CREATE JSON WITH COURSE OFFERINGS FOR EACH QUARTER, ADD THEM TO JSON. THIS JSON WILL BE LATER MERGED WITH FULL COURSE JSON.

In [ ]:
import json

# Load JSONs
with open("course_offerings.json", "r") as f:
    courses_by_quarter = json.load(f)  # {"Fall Quarter 2023": ["CSE 11", "COGS 4", ...]}

with open("courses_and_professors_v2.json", "r") as f:
    courses = json.load(f)  # Course details JSON

# Map quarter names for consistency
quarter_mapping = {
    "Fall Quarter 2023": "fall",
    "Winter Quarter 2024": "winter",
    "Spring Quarter 2024": "spring",
}

# Iterate over each course in the catalog and add offerings
for course in courses:
    course_id = course["course_id"]  # Example: "AAS 10"

    # Initialize offerings list if it doesn't exist
    if "offerings" not in course:
        course["offerings"] = []

    # Check if the course appears in any quarter and update offerings
    for quarter, quarter_key in quarter_mapping.items():
        if course_id in courses_by_quarter.get(quarter, []):
            course["offerings"].append(quarter_key)

# Save updated course catalog
with open("courses_and_professors_v3.json", "w") as f:
    json.dump(courses, f, indent=4)

print("✅ Course catalog updated with offerings!")


TEST CODE TO EXTRACT CATEGORY (lower div, upper div) FROM BIOLOGY PAGE - doesn't currently work.

In [ ]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
from urllib.parse import urlparse
import time
import json

# Set up Selenium
options = webdriver.ChromeOptions()
options.add_argument("--headless")  # Run without opening a browser
driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()), options=options)

# Debugging print
print("🚀 Starting single department scraping test...")

# Target department URL (Biology)
dept_url = "https://catalog.ucsd.edu/courses/BIOL.html"
program_name = "BIOL"  # Extracted manually since we're testing one link

print(f"📂 Accessing department page: {dept_url}")
driver.get(dept_url)
time.sleep(2)  # Let the page load

# Parse HTML content
soup = BeautifulSoup(driver.page_source, "html.parser")
print("✅ Page loaded and parsed.")

# List to store course data
all_courses = []

# Current category tracking
current_category = "other"

# Finding relevant elements (headings and course names)
elements = soup.find_all(["h2", "p"], class_=lambda x: x in ["course-name", "course-descriptions", "anchor-parent", None])

print(f"🔍 Found {len(elements)} potential course elements.")

for el in elements[:30]:
    if el.name == "h2":
        print('entered h2')
        heading_text = el.get_text(strip=True).lower()
        print('heading_text' + heading_text)
        print(f"📌 Found section heading: {heading_text}")

        if "lower division" in heading_text:
            current_category = "lower division"
        elif "upper division" in heading_text:
            current_category = "upper division"
        elif "graduate" in heading_text:
            current_category = "graduate"
        else:
            current_category = "other"

    elif el.name == "p" and "course-name" in el.get("class", []):
        course_text = el.get_text(strip=True)
        print(f"🔹 Found course: {course_text}")

        parts = course_text.split(".", 1)
        if len(parts) < 2:
            print("⚠️ Skipping malformed course entry.")
            continue

        course_id, course_name = parts
        course_id = course_id.strip().replace(" ", "")
        course_name = course_name.strip()

        # Description is typically the next sibling with class="course-descriptions"
        desc_el = el.find_next("p", class_="course-descriptions")
        description = desc_el.get_text(strip=True) if desc_el else "No description available"

        print(f"✅ Adding Course: ID={course_id}, Name={course_name}, Category={current_category}")

        all_courses.append({
            "course_id": course_id,
            "course_name": course_name,
            "description": description,
            "category": current_category,
            "program": program_name  # Manually set since we're testing one department
        })

# Finished scraping
driver.quit()
print("🚪 Selenium driver closed.")

# Save JSON
output_filename = "biol_courses.json"
with open(output_filename, "w") as f:
    json.dump(all_courses, f, indent=4)

print(f"🎉 Saved {len(all_courses)} Biology courses to {output_filename}!")


UPLOAD COURSES TO VECTOR DATABASE

In [25]:
#need to load environment variables
import os
from dotenv import load_dotenv
from pinecone import Pinecone
load_dotenv(override=True)

PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
index_name = os.getenv("PINECONE_INDEX")
#index_name = 'openaicourses'
pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)
# View all vector IDs (up to 100 at a time)
index.describe_index_stats()


{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 7243}},
 'total_vector_count': 7243}

In [ ]:
import pinecone
from pinecone import Pinecone
from sentence_transformers import SentenceTransformer
import pandas as pd
import json
from tqdm import tqdm

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index(index_name)

# Load the embedding model
model = SentenceTransformer("intfloat/multilingual-e5-large")
print("✅ Model loaded with dimension:", model.get_sentence_embedding_dimension())

# Load JSON data into a Pandas DataFrame
json_file_path = "courses_and_professors_v3.json"

with open(json_file_path, "r") as f:
    courses = json.load(f)

df = pd.DataFrame(courses)

# Function to insert courses into Pinecone
def insert_courses_into_pinecone(df):
    vectors = []
    
    for i, row in tqdm(df.iterrows(), total = len(df), desc="Processing Courses"):
        try:
            # Ensure Course ID is valid
            if row.get("course_id"):
                course_id = "".join(c if c.isalnum() else "" for c in row["course_id"])
            else:
                course_id = "NoCourseID"

            #print(f"🔹 Processing Course {i+1}/{len(df)}: {course_id}")

            # Extract Professors' Data
            professors = row.get("professors", [])

            if not isinstance(professors, list):
                #print(f"⚠️ Warning: 'professors' field for {course_id} is not a list. Converting to an empty list.")
                professors = []


            professor_details = [
                {
                    "name": prof.get("name", "Unknown"),
                    "quality_rating": prof.get("quality_rating", "N/A"),
                    "num_ratings": prof.get("num_ratings", "N/A"),
                    "would_take_again": prof.get("would_take_again", "N/A"),
                    "difficulty": prof.get("difficulty", "N/A"),
                    "profile_link": prof.get("profile_link", ""),
                    "department": prof.get("department", "Unknown")
                }
                for prof in professors
            ]
            #print('professor_details: ' + str(professor_details))

            # Create text for embedding (including professor details)
            professor_text = "; ".join([f"{prof['name']} (Rating: {prof['quality_rating']}, Difficulty: {prof['difficulty']})"
                                        for prof in professor_details])
            text_to_embed = f"{row['course_name']}: {row['description']} | Professors: {professor_text}"
            #print('professor_text : ' + str(professor_text))
            #print('text_to_embed : ' + str(text_to_embed))

            # Generate embedding
            embedding = model.encode(text_to_embed).tolist()

            # Prepare metadata
            metadata = {
                "course_name": row["course_name"].strip() if row["course_name"] else "No Course Name",
                "description": row["description"],
                "credits": row.get("credits", "N/A"),
                "prerequisites": row.get("prerequisites", "None"),
                "professors": professor_text # Store full professor details in metadata
            }

            # Add to batch
            vectors.append((course_id, embedding, metadata))
            #print(f"✅ Course {course_id} added to batch.")

        except Exception as e:
            #print(f"❌ Error processing course {i+1}: {e}")
            pass

    # Insert into Pinecone in batches
    batch_size = 100
    for i in range(0, len(vectors), batch_size):
        batch = vectors[i : i + batch_size]
        index.upsert(vectors=batch)
        print(f"📦 Batch {i//batch_size + 1} ({len(batch)} courses) uploaded to Pinecone.")

    print("🚀 Data successfully inserted into Pinecone!")

# Run insertion
insert_courses_into_pinecone(df)

In [ ]:
df